# Topic Modeling — LDA, NMF, and coherence, from scratch

This notebook *proves* every qualitative claim on the page with an `assert` **before** printing the number. It builds a tiny corpus with **planted** topics (so recovery can be checked), runs a from-scratch **collapsed-Gibbs LDA** sampler and a from-scratch **multiplicative-update NMF**, computes **topic coherence** (NPMI / UMass) from scratch, and finishes with the scikit-learn end-to-end on a readable corpus.

Every function and constant is imported from the canonical [`topic_modeling.py`](topic_modeling.py) so the notebook, the page, and the figures cannot drift.

## 0. Setup and device honesty

This chapter is NumPy + scikit-learn, so it runs on **CPU** — we print the device honestly. Everything stochastic is driven by an explicit seed (`seed=7`) through a local `np.random.default_rng`, so results are reproducible.

In [1]:
import numpy as np

# Canonical source of truth: every function and constant comes from the sibling module.
from topic_modeling import (
    SEED, ALPHA, ETA, PLANTED_TOPIC_WORDS, PLANTED_TOPIC_NAMES, READABLE_CORPUS,
    make_planted_corpus, docs_to_count_matrix, GibbsLDA,
    nmf_multiplicative, nmf_top_words, nmf_example_trace,
    npmi_coherence, umass_coherence, purity,
    sweep_k_coherence_perplexity, sklearn_lda_nmf,
)

print('device: cpu (NumPy + scikit-learn)')
print('numpy:', np.__version__)
try:
    import torch
    print('torch:', torch.__version__)
except ImportError:
    print('torch: not importable (not needed — this chapter is NumPy + scikit-learn)')
print('seed:', SEED)

device: cpu (NumPy + scikit-learn)
numpy: 2.4.6


torch: 2.12.0
seed: 7


## 1. A corpus with planted topics

We generate 60 documents over an 18-word vocabulary split into three disjoint themes (sports / cooking / space). Each document has one dominant topic; 15% of its words *leak* from other topics, so it's a genuine admixture. Because we **know** the planting, we can later check the models *recover* it. The bag forgets order, so a document is just a multiset of word ids.

In [2]:
docs, vocab, true_topic = make_planted_corpus()
V = len(vocab)
counts = docs_to_count_matrix(docs, V)

# the three planted vocabularies are disjoint — no word belongs to two planted topics
planted_sets = [set(w) for w in PLANTED_TOPIC_WORDS]
assert set.intersection(*planted_sets) == set(), 'planted topics should be disjoint'
assert counts.shape == (60, 18) and len(docs) == 60

print(f'planted corpus: {len(docs)} docs, |V|={V}')
print('vocabulary:', vocab)
print('planted topics:', dict(zip(PLANTED_TOPIC_NAMES, PLANTED_TOPIC_WORDS)))
print('first document (token ids):', docs[0][:12], '...')

planted corpus: 60 docs, |V|=18
vocabulary: ['bake', 'coach', 'galaxy', 'garlic', 'goal', 'league', 'match', 'moon', 'onion', 'orbit', 'planet', 'recipe', 'rocket', 'simmer', 'striker', 'team', 'telescope', 'tomato']
planted topics: {'sports': ('goal', 'team', 'match', 'league', 'striker', 'coach'), 'cooking': ('garlic', 'onion', 'tomato', 'recipe', 'bake', 'simmer'), 'space': ('orbit', 'planet', 'rocket', 'galaxy', 'telescope', 'moon')}
first document (token ids): [14, 1, 4, 15, 0, 4, 6, 14, 15, 5, 5, 14] ...


## 2. Collapsed-Gibbs LDA, from scratch — does it recover the planting?

The sampler reassigns each word's topic from the conditional $p(z_i=k\mid\cdot)\propto (n_{d,k}+\alpha)\,(n_{k,w}+\eta)/(n_k+V\eta)$ — the exact update on the page. After 300 sweeps we read each topic's top words and check the **doc-topic purity** against the planted truth. We assert purity is essentially perfect *before* printing the topics.

In [3]:
lda = GibbsLDA(n_topics=3, n_iter=300, seed=SEED).fit(docs, V)
pred = lda.doc_topic().argmax(axis=1)
pur = purity(true_topic, pred, 3)

assert pur >= 0.95, f'Gibbs LDA should recover the planted topics (purity={pur:.3f})'

print(f'doc-topic purity vs planted truth: {pur:.3f}')
print('\nrecovered topics (top words):')
for k, words in enumerate(lda.top_words(vocab, n=6)):
    print(f'  topic {k}: ' + ', '.join(words))

doc-topic purity vs planted truth: 1.000

recovered topics (top words):
  topic 0: onion, tomato, bake, garlic, simmer, recipe
  topic 1: rocket, planet, moon, telescope, galaxy, orbit
  topic 2: coach, match, team, striker, goal, league


Each recovered topic should be a permutation of one planted topic's word set (numbering is arbitrary, so we match by overlap). This is the rigorous version of "the topics look right."

In [4]:
recovered = [set(w) for w in lda.top_words(vocab, n=6)]
for planted in PLANTED_TOPIC_WORDS:
    best = max(recovered, key=lambda r: len(set(planted) & r))
    overlap = len(set(planted) & best)
    assert overlap == 6, f'planted topic {planted} not fully recovered (overlap={overlap})'
print('every planted topic recovered exactly (6/6 words matched).')

every planted topic recovered exactly (6/6 words matched).


## 3. NMF by multiplicative updates, from scratch

On the same count matrix, factor $V\approx WH$ with $W,H\ge 0$ via Lee & Seung's updates. We assert the reconstruction error decreases **monotonically** (the convergence guarantee) and that NMF recovers the same three themes.

In [5]:
W, H, errors = nmf_multiplicative(counts, n_topics=3, n_iter=400, seed=SEED)

assert (W >= 0).all() and (H >= 0).all(), 'NMF factors must stay non-negative'
assert all(errors[i + 1] <= errors[i] + 1e-6 for i in range(len(errors) - 1)), 'must be monotone'

print(f'Frobenius error: {errors[0]:.2f} -> {errors[-1]:.2f}  (monotone decrease)')
print('\nNMF recovered topics (top words):')
for k, words in enumerate(nmf_top_words(H, vocab, n=6)):
    print(f'  topic {k}: ' + ', '.join(words))

Frobenius error: 106.92 -> 42.58  (monotone decrease)

NMF recovered topics (top words):
  topic 0: coach, team, match, striker, goal, league
  topic 1: rocket, planet, moon, galaxy, telescope, orbit
  topic 2: onion, tomato, bake, garlic, simmer, recipe


### The 3×3 by-hand example (Example 3 on the page)

The exact little factorization traced by hand — one update drops the error from 7.55 to 4.30, and it keeps falling. These are the page's numbers, reproduced.

In [6]:
trace = nmf_example_trace()
assert trace['errors'][0] == 7.55 and trace['errors'][1] == 4.30
assert all(trace['errors'][i + 1] <= trace['errors'][i] for i in range(len(trace['errors']) - 1))
print('error trajectory:', trace['errors'])
print('H1[0] =', trace['H1_row0'], '  W1[0] =', trace['W1_row0'])

error trajectory: [7.55, 4.3, 4.08, 4.04, 3.91, 3.31, 2.03]
H1[0] = [0.977, 2.087, 1.015]   W1[0] = [0.332, 1.164]


## 4. Topic coherence from scratch — a real topic beats a random word set

Coherence is only trustworthy if it scores genuinely co-occurring words above random ones. We compute NPMI and UMass (from scratch, off document co-occurrence) for a planted topic versus a random word set of the same size, and assert the coherent set wins on both.

In [7]:
coherent = [vocab.index(w) for w in PLANTED_TOPIC_WORDS[0]]   # a real planted topic
rng = np.random.default_rng(SEED)
random_set = list(rng.choice(V, size=len(coherent), replace=False))

c_npmi, r_npmi = npmi_coherence(coherent, counts), npmi_coherence(random_set, counts)
c_umass, r_umass = umass_coherence(coherent, counts), umass_coherence(random_set, counts)

assert c_npmi > r_npmi, 'coherent topic must out-score random on NPMI'
assert c_umass > r_umass, 'coherent topic must out-score random on UMass'

print(f'NPMI : coherent {c_npmi:+.3f}  vs  random {r_npmi:+.3f}')
print(f'UMass: coherent {c_umass:+.3f}  vs  random {r_umass:+.3f}')

NPMI : coherent +0.243  vs  random +0.017
UMass: coherent -0.298  vs  random -0.439


## 5. Choosing K — perplexity vs coherence

Sweep K with scikit-learn LDA on a train/test split of the planted corpus. Held-out perplexity is minimized at the **true K = 3**; coherence is high there too but noisier — the famous divergence. We assert perplexity bottoms at K = 3.

In [8]:
ks, coherences, perplexities = sweep_k_coherence_perplexity(counts, k_values=(2, 3, 4, 5, 6, 8))
best_k = ks[int(np.argmin(perplexities))]

assert best_k == 3, f'held-out perplexity should bottom at the true K=3 (got {best_k})'

print(f"{'K':>3} | {'perplexity':>11} | {'UMass coherence':>16}")
print('-' * 38)
for k, c, p in zip(ks, coherences, perplexities):
    print(f'{k:>3} | {p:>11.2f} | {c:>+16.3f}')
print(f'\nperplexity-best K = {best_k}  (the true number of themes)')

  K |  perplexity |  UMass coherence
--------------------------------------
  2 |       17.78 |           -0.382
  3 |       14.36 |           -0.324
  4 |       15.37 |           -0.316
  5 |       14.91 |           -0.420
  6 |       16.99 |           -0.357
  8 |       17.59 |           -0.416

perplexity-best K = 3  (the true number of themes)


## 6. scikit-learn end-to-end on a readable corpus

Finally, the practical pipeline on the 18-document sports/cooking/space corpus: LDA on raw **counts**, NMF on **TF-IDF** (the two correct representations). Both recover the three themes with no labels, and doc-0 (a match report) loads ~97% on its topic.

In [9]:
lda_topics, nmf_topics, theta0 = sklearn_lda_nmf(READABLE_CORPUS, n_topics=3)

assert theta0.max() > 0.9, 'doc-0 (a match report) should be dominated by one topic'

print('LDA topics (counts):')
for k, words in enumerate(lda_topics):
    print(f'  topic {k}: ' + ', '.join(words))
print('NMF topics (TF-IDF):')
for k, words in enumerate(nmf_topics):
    print(f'  topic {k}: ' + ', '.join(words))
print('doc-0 topic mixture (theta):', np.round(theta0, 3))

LDA topics (counts):
  topic 0: team, goal, football, striker, league, won
  topic 1: garlic, onion, recipe, tomato, basil, eggs
  topic 2: planet, galaxy, moon, spacecraft, stars, orbit
NMF topics (TF-IDF):
  topic 0: team, goal, football, striker, won, scored
  topic 1: onion, garlic, tomato, basil, simmer, sauce
  topic 2: planet, telescope, spacecraft, galaxy, rocket, satellite
doc-0 topic mixture (theta): [0.973 0.014 0.014]


## Recap

- A from-scratch **collapsed-Gibbs LDA** recovered three planted topics at **purity 1.0**.
- A from-scratch **multiplicative-update NMF** recovered the same themes with a **monotone** error decrease.
- **Coherence** (NPMI / UMass) scored a real topic far above a random word set.
- Held-out **perplexity** bottomed at the true K = 3, while coherence was noisier — read the topics.
- scikit-learn **LDA (counts)** and **NMF (TF-IDF)** recovered sports/cooking/space with no labels.

Every claim was asserted before it was printed — nothing here is hand-typed.